# 底盘 EKF 与异步运动

默认使用内存模拟器，不连接硬件。基于 odom 位姿和 IMU 角速度做平面 EKF，输出 `graphmap.pose.Pose`；`move` 以米为单位、后退为负，`rotate` 接受角度或弧度、左转为正。

坐标统一为 X 前 / Y 左 / Z 上，`T_odom_body` 将车体坐标变到 odom。示例覆盖前进、后退、左右转、时间戳查询和相对 Pose 变换。安装 graphmap（Pose v3）、RSim 及 scipykit 后，在已有事件循环中直接 `await`。

In [1]:
import asyncio
import contextlib
import json
import os
from pathlib import Path
import sys

import numpy as np
from graphmap.pose import Pose
from rsim import Runtime, Chassis, SSHConfig, VelocityCommand
from rsim.motion import ChassisController, MotionError

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "rsim").is_dir() and (p / "pyproject.toml").exists())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from examples.chassis_motion import SimulatedChassis

ASSETS = ROOT / "assets" / "motion"
ASSETS.mkdir(parents=True, exist_ok=True)
os.environ["NOTEBOOK_ASSETS_ROOT"] = str(ROOT / "assets")
os.environ["NOTEBOOK_NAME"] = "motion"
from scipykit.mtp_initializer import subplots, label_axes, disp, plt

## 模拟闭环运动

只有这里的内存模拟器开启 `motion_enabled=True`。模拟时间加速 4 倍，IMU 带零偏及噪声；记录协程与运动控制并发执行。每次运动返回最终 Pose，成功、异常及取消都归零。

In [ ]:
sim = SimulatedChassis(speedup=4, noise=True)
robot = ChassisController(sim, motion_enabled=True, hz=50, max_linear=.3, max_angular=1.)
samples = []

async def record():
    previous = 0
    while True:
        estimate = await robot.odometry.estimate.get(after=previous, timeout=2)
        previous = estimate.sequence
        truth = (await sim.truth.get()).data
        raw = (await sim.odom.get()).data["pose"]["pose"]["position"]
        pose = estimate.data["pose"]
        samples.append([estimate.stamp_ns * 1e-9, *truth.position[:2],
                        raw["x"], raw["y"], *pose.position[:2],
                        pose.euler_rad[2], estimate.data["gyro_bias"]])

async with Runtime(robot):
    first = await robot.pose.get(timeout=5)
    assert await robot.pose.get(timestamp_ns=first.stamp_ns, clock=first.clock) is first
    recorder = asyncio.create_task(record())
    try:
        await asyncio.sleep(.2)
        forward = await robot.move(.30)
        reverse = await robot.move(-.15)
        left = await robot.rotate(yaw_deg=90)
        right = await robot.rotate(yaw_rad=-np.pi / 2)
        await asyncio.sleep(.2)
        relative = (~first.data) * right
        print("Final T_odom_body:", right)
        print("Relative translation (m):", relative.position)
        print("Relative yaw (deg):", relative.euler[2])
        print("Samples recorded concurrently:", len(samples))
    finally:
        recorder.cancel()
        with contextlib.suppress(asyncio.CancelledError):
            await recorder

samples = np.asarray(samples)
commands = np.array([[t, c.linear_x, c.angular_z] for t, c in sim.commands])
assert np.all(commands[-1, 1:] == 0)
np.savez(ASSETS / "simulation.npz", samples=samples, commands=commands)
print("Final command:", sim.commands[-1][1])

In [ ]:
fig, axes = subplots(1, 3, figsize=(12, 3.4), layout="constrained")
axes[0].plot(samples[:, 1], samples[:, 2], label="truth")
axes[0].scatter(samples[:, 3], samples[:, 4], s=3, alpha=.3, label="wheel odom")
axes[0].plot(samples[:, 5], samples[:, 6], label="EKF")
axes[0].set_aspect("equal", adjustable="datalim")
label_axes(axes[0], "odom x (m)", "odom y (m)", grid=True, legend=True)
axes[1].plot(samples[:, 0] - samples[0, 0], np.rad2deg(np.unwrap(samples[:, 7])))
label_axes(axes[1], "simulation time (s)", "yaw (deg)", grid=True)
axes[2].plot(commands[:, 0] - commands[0, 0], commands[:, 1], label="linear (m/s)")
axes[2].plot(commands[:, 0] - commands[0, 0], commands[:, 2], label="angular (rad/s)")
label_axes(axes[2], "wall time (s)", "command", grid=True, legend=True)
disp(fig, "simulation")
plt.close(fig)

## 可选实机：仅采集位姿、发送零速

默认跳过。将 `RSIM_CHASSIS_CONFIG` 环境变量设为本地 JSON 配置路径，然后重新执行此单元格。配置格式：

```json
{
  "ssh": {"host": "<ssh-alias>", "python": "python2", "setup": ["<ros-setup.bash>", "<workspace-setup.bash>"]},
  "topics": {"imu_topic": "<imu-topic>", "odom_topic": "<raw-wheel-odom-topic>", "cmd_vel_topic": "<velocity-topic>"},
  "T_body_imu": {"translation": [0, 0, 0], "rotation": [0, 0, 0, 1], "wrd_frame": "<odom-child-frame>", "ego_frame": "<imu-frame>"}
}
```

外参应从设备 TF 或标定取得，不应直接采用上面的单位旋转示意。此单元格固定 `motion_enabled=False`，只调用 `move(0)` / `rotate(0)` / `stop()`；没有非零实机运动演示。

In [6]:
config_path = os.environ.get("RSIM_CHASSIS_CONFIG")
config_path = "/home/root/Projects/RSim/assets/motion/hardware-config.json"
if not config_path:
    print("Hardware skipped: RSIM_CHASSIS_CONFIG is unset.")
else:
    config = json.loads(Path(config_path).read_text())
    chassis = Chassis(
        SSHConfig(**config["ssh"]),
        **config["topics"],
        log_path=ASSETS / "agent.log"
        )
    hardware = ChassisController(
        chassis,
        T_body_imu=Pose(**config["T_body_imu"]),
        motion_enabled=True
        )
    poses = []
    async with Runtime(hardware):
        frame = await hardware.pose.get(timeout=15)
        for _ in range(30):
            frame = await hardware.pose.get(
                after=frame.sequence, timeout=3
                )
            poses.append(frame.data.matrix)
        await hardware.move(0.5, timeout=10)
        await hardware.rotate(yaw_deg=90, timeout=10)
        await hardware.stop()
        feedback = await chassis.velocity_feedback.get(timeout=3)
        assert feedback.data["command"] == VelocityCommand()
        report = {
            "frames": len(poses),
            "clock": frame.clock,
            "world": frame.data.wrd_frame,
            "body": frame.data.ego_frame,
            "last_pose": frame.data.matrix.tolist(),
            "motion_enabled": hardware.motion_enabled,
            "zero_ack": feedback.data["ack"]
            }
    np.save(ASSETS / "hardware-poses.npy", np.asarray(poses))
    (ASSETS / "hardware.json").write_text(json.dumps(report, indent=2))
    print(report)

{'frames': 30, 'clock': 'ros1:RSim', 'world': 'odom', 'body': 'base_footprint', 'last_pose': [[0.9556147181804158, -0.2946192634519414, 0.0, 0.6439790725708008], [0.2946192634519414, 0.9556147181804158, 0.0, 0.1671706885099411], [0.0, 0.0, 1.0, 0.0], [0.0, 0.0, 0.0, 1.0]], 'motion_enabled': True, 'zero_ack': {'connections': 3, 'published': True}}


`robot.pose.get()` 支持按时间戳回查；`robot.odometry.estimate` 还包含协方差、速度、角速度和陀螺仪零偏。计算量增加时，可将估计器放入 `ProcessPlacement`，控制器留在 Notebook 所在进程，见 [接口说明](../docs/motion.md)。

这是平面相对运动原型，没有避障或全局定位；实际底盘运动精度尚需单独标定与运动验证。本 Notebook 的实机路径始终只发零速。